# 🧹 16 · Denoising and Super-Resolution
### Restoring signal from noise, and detail from downsampling — from a mean filter to a convolutional denoiser

**Part H · Restoring signal** · [⟵ 15 Image registration](15_image_registration.ipynb) · [🏠 Course home](../README.md) · [17 Restoration & inverse problems ⟶](17_restoration_and_inverse_problems.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/16_denoising_and_super_resolution.ipynb)

| | |
|---|---|
| **Hardware target** | CPU; the two small PyTorch trainings together take roughly 1–2 minutes |
| **Core frameworks** | NumPy, SciPy, PyTorch (tiny CNNs, CPU-only), PyWavelets, Matplotlib |
| **Prerequisites** | Module 03 (filtering), Module 04 (frequency domain), Module 08 (deep learning basics) |
| **Estimated time** | 80–100 minutes |
| **02-740 topics covered** | noise models (Gaussian, impulse, uniform, periodic), denoising (mean filter, FFT, wavelet transform, convolutional denoising autoencoder), denoising quality measures (PSNR, SSIM), super-resolution (bicubic baseline, SRCNN) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images, noise and training data in this notebook are SIMULATED by code — nothing is downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- recognise and generate the four classic noise models — Gaussian, impulse (salt-and-pepper), uniform, and periodic — and explain what physical process each one models;
- apply and compare four denoising strategies — the linear mean filter, FFT low-pass filtering, wavelet-coefficient thresholding, and a trained convolutional denoising autoencoder — and quantify each with PSNR and SSIM;
- explain the fundamental limit on single-image upscaling ("you cannot add information that was never captured") and how a learned prior (SRCNN) gets around it in practice;
- clearly distinguish **CNN-based super-resolution** (a learned image-to-image mapping) from **single-molecule-localization super-resolution microscopy** (PALM/STORM: beating the diffraction limit by precisely localising many sparse, individually-resolvable emitters) — two "super-resolution" techniques that solve completely different problems.

## 🗺️ Contents

1. Four noise models
2. Denoising methods: mean filter, FFT, wavelet thresholding, and a learned denoiser
3. Denoising quality measures: PSNR and SSIM
4. Super-resolution: the bicubic baseline and SRCNN
5. Two different "super-resolutions": CNN upscaling vs. single-molecule localisation microscopy

## 🧫 The biological question

Every photon that reaches a microscope's detector has fought its way through shot noise, read noise, and often very little signal to begin with — live-cell imaging in particular demands low laser power (to avoid phototoxicity and photobleaching), which means a worse signal-to-noise ratio than a fixed, heavily-illuminated sample would tolerate. **Denoising** asks: given this one noisy observation, what is our best estimate of the clean signal underneath it? And because many biological structures (synaptic vesicles, viral particles, cytoskeletal filaments) are smaller than a conventional microscope's diffraction limit (~200–250 nm), a complementary question arises: how much extra detail can we recover beyond what the optics nominally allow — and by what mechanism? This notebook builds both restoration and super-resolution tools from scratch, and is careful to show that "super-resolution" means two genuinely different things in these two contexts.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch pywavelets
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
import pywt
SEED = 16
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_nuclei_image`, the noise primitives `add_gaussian_noise` / `add_salt_pepper_noise` / `add_periodic_noise`, `fft_lowpass_denoise`, `psnr`, `simple_ssim`, `fit_gaussian_spot` (for the PALM/STORM toy demo), `simulate_point_pattern`, and `show_image_grid`.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def add_salt_pepper_noise(image: np.ndarray, amount: float, rng: np.random.Generator) -> np.ndarray:
    """Impulse noise: a random ``amount`` fraction of pixels are slammed to 0 or 1."""
    out = image.copy()
    mask = rng.random(image.shape) < amount
    salt = rng.random(image.shape) < 0.5
    out[mask & salt] = image.max()
    out[mask & ~salt] = image.min()
    return out


def add_periodic_noise(image: np.ndarray, amplitude: float, freq: tuple[float, float]) -> np.ndarray:
    """Periodic (structured) noise: a sinusoidal stripe pattern, as from scan-line pickup."""
    h, w = image.shape
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    return image + amplitude * np.sin(2 * np.pi * (freq[0] * yy / h + freq[1] * xx / w))


def fft_lowpass_denoise(image: np.ndarray, cutoff_frac: float = 0.15) -> np.ndarray:
    """Denoise by zeroing high spatial frequencies in the Fourier domain (an ideal low-pass
    filter at ``cutoff_frac`` of the Nyquist frequency), then inverse-transforming."""
    f = np.fft.fftshift(np.fft.fft2(image))
    h, w = image.shape
    yy, xx = np.meshgrid(np.arange(h) - h / 2, np.arange(w) - w / 2, indexing="ij")
    radius = np.hypot(yy, xx) / (min(h, w) / 2)
    f[radius > cutoff_frac] = 0
    return np.real(np.fft.ifft2(np.fft.ifftshift(f)))


def psnr(reference: np.ndarray, estimate: np.ndarray, data_range: float | None = None) -> float:
    """Peak signal-to-noise ratio (dB) between a reference and an estimate; higher is better."""
    data_range = data_range or (reference.max() - reference.min() + 1e-9)
    mse = np.mean((reference - estimate) ** 2)
    if mse <= 1e-12:
        return float("inf")
    return 10 * np.log10(data_range ** 2 / mse)


def simple_ssim(a: np.ndarray, b: np.ndarray, data_range: float | None = None) -> float:
    """A simplified, global (whole-image, not windowed) structural-similarity index: combines
    luminance, contrast and structure agreement; 1.0 = identical. For a full windowed SSIM use
    ``skimage.metrics.structural_similarity``."""
    data_range = data_range or (max(a.max(), b.max()) - min(a.min(), b.min()) + 1e-9)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_a, mu_b = a.mean(), b.mean()
    var_a, var_b = a.var(), b.var()
    cov = ((a - mu_a) * (b - mu_b)).mean()
    return float(((2 * mu_a * mu_b + c1) * (2 * cov + c2)) /
                 ((mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2)))


def fit_gaussian_spot(patch: np.ndarray):
    """Sub-pixel localisation of one bright spot: fit a 2-D Gaussian's centroid by intensity-
    weighted moments (the simple, closed-form version of 'Gaussian fitting' spot localisation)."""
    yy, xx = np.meshgrid(np.arange(patch.shape[0]), np.arange(patch.shape[1]), indexing="ij")
    w = np.clip(patch - patch.min(), 0, None)
    total = w.sum()
    if total <= 0:
        return patch.shape[0] / 2, patch.shape[1] / 2
    cy = (w * yy).sum() / total
    cx = (w * xx).sum() / total
    return cy, cx


def simulate_point_pattern(n_points: int = 150, size: float = 100.0,
                            pattern: str = "random", cluster_std: float = 4.0,
                            n_clusters: int = 8, min_dist: float = 0.0,
                            seed: int = 0) -> np.ndarray:
    """SIMULATED 2-D point coordinates with a known spatial ``pattern``.

    ``pattern`` is one of ``"random"`` (complete spatial randomness), ``"clustered"``
    (Gaussian clumps around ``n_clusters`` parents) or ``"dispersed"`` (points repel each
    other, enforcing roughly ``min_dist``). Returns an (n_points, 2) array; the pattern name
    is the ground truth used in the 🧪 checks.
    """
    rng = np.random.default_rng(seed)
    if pattern == "random":
        pts = rng.uniform(0, size, size=(n_points, 2))
    elif pattern == "clustered":
        centers = rng.uniform(size * 0.15, size * 0.85, size=(n_clusters, 2))
        choice = rng.integers(0, n_clusters, size=n_points)
        pts = centers[choice] + rng.normal(0, cluster_std, size=(n_points, 2))
        pts = np.clip(pts, 0, size)
    elif pattern == "dispersed":
        min_dist = min_dist or size / np.sqrt(n_points) * 0.9
        pts = []
        attempts = 0
        while len(pts) < n_points and attempts < n_points * 200:
            attempts += 1
            cand = rng.uniform(0, size, size=2)
            if all(np.hypot(*(cand - p)) > min_dist for p in pts):
                pts.append(cand)
        pts = np.array(pts)
    else:
        raise ValueError(f"unknown pattern {pattern!r}")
    return pts


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · Four noise models

A noisy image is usually modelled as $y = x + n$ (or, for shot noise, a multiplicative/Poisson process) where $x$ is the true signal and $n$ is noise with some characteristic statistical distribution. Four distributions recur constantly in bioimaging:

- **Gaussian noise** — the classic camera read-noise floor: every pixel gets an independent draw from $\mathcal{N}(0, \sigma^2)$, evenly spread across the whole image.
- **Impulse (salt-and-pepper) noise** — a random subset of pixels are slammed to the extreme minimum or maximum value, looking like sparse black-and-white "sparks"; it can arise from dead/hot sensor pixels or transmission errors.
- **Uniform noise** — each pixel gets an independent draw from a flat distribution on $[a, b]$; visually more "even" than impulse noise, without impulse noise's sparse extreme spikes.
- **Periodic noise** — a repeating sinusoidal pattern, typically from electrical or electromechanical interference during acquisition (e.g. scan-line pickup); it looks like stripes superimposed on the image, and — crucially — shows up as a small number of sharp peaks in the image's Fourier spectrum (Module 04), which is exactly what makes it easy to remove there.

In [ ]:
# ── 1.1 Generate a clean reference image and all four noise types ──
def add_uniform_noise(image, low, high, rng):
    """Uniform noise: each pixel gets an independent draw from U(low, high) — this
    notebook's own helper; not in the shared toolkit since it is used only here."""
    return image + rng.uniform(low, high, size=image.shape)

sim = simulate_nuclei_image(n_cells=14, size=128, noise_sigma=0.0, seed=SEED)
clean = sim["image"]

noisy_gaussian = add_gaussian_noise(clean, 0.12, rng)
noisy_impulse = add_salt_pepper_noise(clean, 0.06, rng)
noisy_uniform = add_uniform_noise(clean, -0.15, 0.15, rng)
noisy_periodic = add_periodic_noise(clean, 0.25, freq=(6.0, 9.0))

show_image_grid([clean, noisy_gaussian, noisy_impulse, noisy_uniform, noisy_periodic],
                 ["clean", "+ Gaussian", "+ impulse", "+ uniform", "+ periodic"], ncols=5)

🔍 **What just happened?** All five images show the *same* underlying simulated cells; only the noise differs. Gaussian and uniform noise both spread evenly across the image (differing mainly in their tails), impulse noise produces sparse, extreme black/white specks, and periodic noise is visually a repeating stripe pattern — all four are common in real microscopy, often simultaneously.

✏️ **Your turn.** Compute `np.fft.fftshift(np.abs(np.fft.fft2(noisy_periodic)))` and display it with `plt.imshow(..., cmap="inferno")`. You should see a small number of sharp bright spots away from the centre, at the spatial frequency of the stripe pattern — exactly the "frequency spikes" the deck's periodic-noise discussion points to as the giveaway for this noise type.

## 2 · Denoising methods

We compare four denoising strategies, in increasing order of sophistication, on the same Gaussian-noise case:

1. **Linear mean filter** — replace every pixel with the average of its neighbourhood. Cheap and effective against Gaussian noise, but it convolves away edges along with the noise, over-smoothing the image.
2. **FFT low-pass filtering** — Module 04's frequency-domain tool: zero out high spatial frequencies (where most noise energy lives) and inverse-transform. Particularly effective against periodic noise (zero out just the offending frequency peaks) and reasonably effective against broadband noise, at the same edge-blurring cost as the mean filter.
3. **Wavelet-coefficient thresholding** — the wavelet transform (Module 04's multiscale idea) decomposes the image into *localised*, multiscale detail coefficients rather than global frequencies. Noise tends to produce many small-magnitude detail coefficients, while real structure produces a few large ones, so **thresholding** the detail coefficients (shrinking or zeroing the small ones, built as "hard" keep-or-kill or "soft" shrink-or-kill rules) can, in principle, remove noise while treating fine, localised structure (like edges) differently from a single global frequency cutoff. How much that pays off in practice depends heavily on the threshold-selection rule and on how much genuinely multiscale detail the image contains — Section 2.5 measures it rather than assuming it.
4. **A convolutional denoising autoencoder (CAE)** — rather than hand-designing a filter, *learn* a mapping from noisy to clean images directly from paired examples, exploiting whatever structure is common across many simulated cell images.

In [ ]:
# ── 2.1 Method 1: linear mean filter ──
denoised_mean = ndi.uniform_filter(noisy_gaussian, size=5)

# ── 2.2 Method 2: FFT low-pass denoising ──
denoised_fft = fft_lowpass_denoise(noisy_gaussian, cutoff_frac=0.22)

# ── 2.3 Method 3: wavelet-coefficient soft thresholding ──
def wavelet_denoise_soft(image, wavelet="db2", level=2, k=3.0):
    """VisuShrink-style wavelet denoising: decompose, soft-threshold the detail coefficients
    at every level with a universal threshold T = k * sigma * sqrt(2 log N), reconstruct."""
    coeffs = pywt.wavedec2(image, wavelet, level=level)
    approx, details = coeffs[0], coeffs[1:]
    sigma = np.median(np.abs(details[-1][0])) / 0.6745 + 1e-9  # robust noise-sigma estimate
    n = image.size
    thresh = k * sigma * np.sqrt(2 * np.log(n))
    soft = lambda c: np.sign(c) * np.maximum(np.abs(c) - thresh, 0)
    new_details = [tuple(soft(sub) for sub in level_coeffs) for level_coeffs in details]
    return pywt.waverec2([approx] + new_details, wavelet)[:image.shape[0], :image.shape[1]]

denoised_wavelet = wavelet_denoise_soft(noisy_gaussian)

show_image_grid([clean, noisy_gaussian, denoised_mean, denoised_fft, denoised_wavelet],
                 ["clean", "noisy", "mean filter", "FFT low-pass", "wavelet soft-threshold"], ncols=5)

🔍 **What just happened?** All three classical methods visibly reduce the noise. Section 2.5 will put numbers on exactly how well each one does on held-out data — and the ranking is worth checking empirically rather than assuming: our simple universal-threshold wavelet rule is a blunt instrument, and on smooth, low-texture synthetic blobs like these it does not automatically beat a well-tuned mean filter or FFT cutoff. Wavelet thresholding's real advantage shows up more on images with sharper, more localised texture, and more sophisticated threshold-selection rules (BayesShrink, SURE) typically close this gap — a reminder that "more sophisticated algorithm" does not automatically mean "better result" without matching the method to the data and tuning it properly.

In [ ]:
# ── 2.4 Method 4: a small convolutional denoising autoencoder, trained from scratch ──
class DenoisingCAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

def make_training_pair(seed, size=64, n_cells=8, noise_sigma=0.12):
    s = simulate_nuclei_image(n_cells=n_cells, size=size, noise_sigma=0.0, seed=seed)
    clean_img = s["image"]
    noisy_img = add_gaussian_noise(clean_img, noise_sigma, np.random.default_rng(seed + 777))
    return noisy_img.astype(np.float32), clean_img.astype(np.float32)

train_pairs = [make_training_pair(seed=100 + i) for i in range(12)]
X_train = torch.tensor(np.stack([p[0] for p in train_pairs]))[:, None, :, :]
Y_train = torch.tensor(np.stack([p[1] for p in train_pairs]))[:, None, :, :]

cae = DenoisingCAE()
optimizer = torch.optim.Adam(cae.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
for epoch in range(150):
    optimizer.zero_grad()
    loss = loss_fn(cae(X_train), Y_train)
    loss.backward()
    optimizer.step()
print(f"final training MSE: {loss.item():.5f}")

# evaluate on a held-out noisy image (different seed, never seen during training)
test_noisy, test_clean = make_training_pair(seed=999)
with torch.no_grad():
    denoised_cae = cae(torch.tensor(test_noisy)[None, None]).numpy()[0, 0]

🧪 **Ground-truth check.** `test_noisy`/`test_clean` were generated with `seed=999`, which never appeared among the 12 training seeds (100–111) — so this is a genuine held-out evaluation, not a measurement of how well the network memorised its training set.

In [ ]:
# ── 2.5 Quantify all four methods on the held-out image with PSNR and SSIM ──
test_mean = ndi.uniform_filter(test_noisy, size=5)
test_fft = fft_lowpass_denoise(test_noisy, cutoff_frac=0.22)
test_wavelet = wavelet_denoise_soft(test_noisy)

for name, estimate in [("noisy (no denoising)", test_noisy), ("mean filter", test_mean),
                       ("FFT low-pass", test_fft), ("wavelet soft-threshold", test_wavelet),
                       ("denoising CAE", denoised_cae)]:
    print(f"{name:24s}  PSNR = {psnr(test_clean, estimate):5.2f} dB   "
          f"SSIM = {simple_ssim(test_clean, estimate):.3f}")

🔍 **What just happened?** PSNR and SSIM (Section 3 explains both properly) let us put a number on "how much better" each method did, on data the CAE never trained on. Expect the CAE to come out ahead of the hand-designed filters here — it was trained specifically to map this exact noise distribution back to clean simulated nuclei, so it can exploit structure (e.g. "blobs are round-ish and smooth") that a generic filter cannot.

### 📘 Lecture notes — why a *denoising* autoencoder, specifically

A plain autoencoder trained to reconstruct its own input can simply learn the identity function and tell you nothing useful. Forcing the network to map a *corrupted* input to the *clean* target (as we just did) prevents that shortcut and forces the network to learn which parts of the input are signal worth keeping and which parts are noise worth discarding — exactly the representation-learning principle behind denoising autoencoders (Vincent et al. 2008). Stacking several such layers, and sharing convolutional weights across spatial locations (rather than a fully-connected layer per pixel), is what turns a plain denoising autoencoder into a *convolutional* one: far fewer parameters, and a built-in assumption that the same local noise-removal rule should apply everywhere in the image.

✏️ **Your turn.** Increase `noise_sigma` in `make_training_pair` to `0.25` (both for training and for the held-out test image) and retrain. Does the CAE's PSNR/SSIM advantage over the classical filters grow or shrink as the noise gets heavier? What does that suggest about when a learned denoiser is worth the extra complexity of training one?

## 3 · Denoising quality measures: PSNR and SSIM

Comparing denoised images by eye is useful but subjective — one person's "good enough" is another's "still too noisy." Two standard numerical measures:

$$\text{PSNR} = 10\log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}}\right), \qquad \text{MSE} = \frac{1}{MN}\sum_{i,j}\big(f(i,j)-g(i,j)\big)^2$$

*In words:* PSNR compares the maximum possible signal value against the mean squared pixel-wise error — higher is better, and it is reported in decibels because images can span a very wide dynamic range. Its main weakness, already visible in Section 2, is that it treats every pixel independently and has no notion of *structure*: an image that is uniformly slightly brighter everywhere and an image with the same MSE concentrated into one badly-blurred edge can get identical PSNR scores despite looking very differently "wrong." **SSIM** (structural similarity) addresses this by comparing luminance, contrast, and structural correlation separately between two images, which tends to track human perceptual judgements of image quality more closely than PSNR alone — this is also why, historically, SSIM has been used not just to *measure* restoration quality but as a *loss function* to directly train restoration networks (since minimising MSE/maximising PSNR alone, as plain SRCNN does in Section 4, does not always produce the perceptually best result).

🧪 **Ground-truth check.** Confirm that `psnr(test_clean, test_clean)` returns `inf` (zero error against itself) and `simple_ssim(test_clean, test_clean)` returns almost exactly `1.0` — these are the sanity extremes both measures must hit by construction.

In [ ]:
print("PSNR(clean, clean) =", psnr(test_clean, test_clean))
print("SSIM(clean, clean) =", simple_ssim(test_clean, test_clean))

## 4 · Super-resolution: the bicubic baseline and SRCNN

Given a small, low-resolution image, how do you make it bigger *without inventing detail that wasn't there*? The honest answer is: you cannot, not purely from that one image. **Data-processing inequality**: no amount of post-processing can add information that was never captured in the first place — simple upscaling methods (nearest-neighbour, bilinear, bicubic interpolation) can only interpolate smoothly between the pixels that exist, which is why they look blurry when pushed to a large magnification factor.

The way around this apparent dead end is to supply **additional information from outside the single image** — specifically, a large collection of *other* high-resolution images, from which a model learns what plausible fine detail tends to look like for this general class of images (e.g. "cell nuclei commonly have smoothly-curved boundaries and slightly grainy interior texture"), then uses that learned prior to "hallucinate" detail when upscaling a *new* low-resolution image. **SRCNN** (super-resolution with a convolutional neural network) does exactly this: bicubic-upsample the low-resolution input first (a cheap, information-free step purely to get the array to the right output size), then train a CNN to map that blurry bicubic result onto the true high-resolution image, learning the residual detail from many training pairs.

In [ ]:
# ── 4.1 Build low/high-resolution training pairs and the bicubic baseline ──
def make_sr_pair(seed, hr_size=96, n_cells=10, downscale=2):
    s = simulate_nuclei_image(n_cells=n_cells, size=hr_size, noise_sigma=0.01, seed=seed)
    hi_res = s["image"].astype(np.float32)
    lo_res = ndi.zoom(hi_res, 1 / downscale, order=1)          # simulate a lower-resolution capture
    bicubic = ndi.zoom(lo_res, downscale, order=3)              # information-free upscale
    bicubic = bicubic[:hi_res.shape[0], :hi_res.shape[1]]
    return lo_res, bicubic, hi_res

sr_train = [make_sr_pair(seed=200 + i) for i in range(10)]
X_bicubic = torch.tensor(np.stack([p[1] for p in sr_train]))[:, None, :, :]
Y_hires = torch.tensor(np.stack([p[2] for p in sr_train]))[:, None, :, :]

class TinySRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 9, padding=4), nn.ReLU(),
            nn.Conv2d(32, 16, 5, padding=2), nn.ReLU(),
            nn.Conv2d(16, 1, 5, padding=2),
        )

    def forward(self, x):
        return x + self.net(x)   # residual learning: predict the DETAIL missing from bicubic

srcnn = TinySRCNN()
optimizer = torch.optim.Adam(srcnn.parameters(), lr=1e-3)
for epoch in range(300):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(srcnn(X_bicubic), Y_hires)
    loss.backward()
    optimizer.step()
print(f"final SRCNN training MSE: {loss.item():.5f}")

In [ ]:
# ── 4.2 Evaluate on a held-out scene ──
lo_test, bicubic_test, hi_test = make_sr_pair(seed=999)
with torch.no_grad():
    srcnn_test = srcnn(torch.tensor(bicubic_test[None, None])).numpy()[0, 0]

print(f"Bicubic alone: PSNR = {psnr(hi_test, bicubic_test):.2f} dB, SSIM = {simple_ssim(hi_test, bicubic_test):.3f}")
print(f"SRCNN        : PSNR = {psnr(hi_test, srcnn_test):.2f} dB, SSIM = {simple_ssim(hi_test, srcnn_test):.3f}")

show_image_grid([lo_test, bicubic_test, srcnn_test, hi_test],
                 ["low-res input", "bicubic baseline", "SRCNN", "true high-res"], ncols=4)

🔍 **What just happened?** SRCNN starts from the *same* blurry bicubic image and should improve both PSNR and SSIM relative to it, on a held-out scene (`seed=999`) never seen in training — because it learned, from the 10 training scenes, what kind of fine detail typically separates a crisp cell-nucleus image from its blurred bicubic upscale, and applies that learned correction here.

✏️ **Your turn.** Increase `downscale` to `3` in `make_sr_pair` (both train and test) and retrain. Super-resolving from a smaller low-resolution input is a harder, more ill-posed problem — does SRCNN's improvement over the bicubic baseline grow, shrink, or stay about the same?

## 5 · Two different "super-resolutions": CNN upscaling vs. single-molecule localisation microscopy

It is easy to conflate "super-resolution" as used in Section 4 with "super-resolution microscopy" (PALM, STORM and relatives) — the terms share a name but solve genuinely different problems:

- **CNN-based super-resolution (Section 4)** starts from *one* already-acquired, low-resolution image and uses a *learned prior* over many other images to hallucinate plausible extra detail. It does not, and cannot, recover information that is truly absent from the optical signal — it is making an informed guess, not a measurement.
- **Single-molecule-localisation microscopy (PALM/STORM)** instead exploits a clever experimental trick: individual fluorophores are made to blink on and off (or are otherwise sparsely activated) so that, in any one camera frame, only a handful of *well-separated* emitters are lit up at once. Each isolated emitter's image is still a diffraction-blurred spot (the same PSF-convolution Module 02 introduced) — but because it does not overlap with its neighbours, its *centre* can be localised to a small fraction of the blur width by fitting a 2D Gaussian (exactly `fit_gaussian_spot`, Module 05), far more precisely than the blur radius itself would suggest. Repeating this over thousands of blink cycles and compositing all the recovered sub-pixel positions reconstructs an image whose **effective resolution is set by the localisation precision, not by the diffraction limit** — a fundamentally different mechanism from learned upscaling.

In [ ]:
# ── 5.1 A minimal PALM/STORM-style toy demo ──
true_emitters = simulate_point_pattern(n_points=18, size=100.0, pattern="dispersed", seed=SEED)
psf_sigma = 2.2  # diffraction-limited blur, in pixels

def render_emitters(points, size=100, sigma=2.2, noise=0.01, seed=0):
    canvas = np.zeros((size, size))
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    for y, x in points:
        canvas += np.exp(-(((yy - y) ** 2 + (xx - x) ** 2) / (2 * sigma ** 2)))
    return add_gaussian_noise(canvas, noise, np.random.default_rng(seed))

diffraction_limited = render_emitters(true_emitters, sigma=psf_sigma, seed=SEED)

# Localise each (well-separated) emitter to sub-pixel precision with a local Gaussian fit
recovered = []
half_win = int(round(3 * psf_sigma))
for y, x in true_emitters:
    y0, x0 = int(round(y)), int(round(x))
    patch = diffraction_limited[max(0, y0 - half_win):y0 + half_win,
                                 max(0, x0 - half_win):x0 + half_win]
    cy, cx = fit_gaussian_spot(patch)
    recovered.append((max(0, y0 - half_win) + cy, max(0, x0 - half_win) + cx))
recovered = np.array(recovered)

localisation_error = np.hypot(*(recovered - true_emitters).T)
print(f"Diffraction-limited blur width (sigma): {psf_sigma:.2f} px")
print(f"Mean localisation error after Gaussian fitting: {localisation_error.mean():.3f} px")
print(f"Localisation precision is ~{psf_sigma / max(localisation_error.mean(), 1e-6):.1f}x "
      f"finer than the blur width — this is the PALM/STORM mechanism, in miniature.")

🧪 **Ground-truth check.** `simulate_point_pattern(..., pattern="dispersed", ...)` enforces a minimum separation between emitters, so each one's PSF-blurred spot in `diffraction_limited` does not overlap its neighbours — exactly the "sparse activation" assumption PALM/STORM relies on. The mean localisation error should be a small fraction of `psf_sigma`, demonstrating resolution well beyond the nominal blur width — achieved by precise localisation of isolated emitters, with **no CNN, no learned prior, and no training data** involved at all.

### 📘 Lecture notes — why this only works when emitters are sparse

The entire localisation-microscopy trick depends on isolating one emitter's blurred spot from its neighbours so that `fit_gaussian_spot` can see a single, clean peak; if two emitters' PSFs overlap substantially, their combined intensity pattern can no longer be unambiguously attributed to two separate, precisely-located sources without a considerably more complex multi-emitter fitting model. This is exactly why real PALM/STORM data requires engineering the *sample* (photoswitchable or blinking fluorophores) so that only a sparse, random subset of molecules is ever lit up in any single camera frame — the hard experimental problem is making sparsity happen, not the localisation math itself.

## 📌 Key takeaways

- Gaussian, impulse, uniform and periodic noise are four distinct statistical models for corruption, each suggesting a different removal strategy (periodic noise especially gives itself away in the Fourier domain).
- Classical denoising filters (mean, FFT low-pass, wavelet-coefficient thresholding) all trade off noise removal against detail preservation in different ways; which one wins is an empirical question that depends on the image content and on how well each method is tuned, not a fixed ranking — a trained convolutional denoising autoencoder can do better than all of them by exploiting learned structure specific to the image class, at the cost of needing training data and compute.
- **PSNR** and **SSIM** quantify denoising/restoration quality numerically, with SSIM tracking structural, perceptual differences that pure pixelwise PSNR can miss.
- Single-image super-resolution cannot add information that was never captured; SRCNN instead learns a prior from many *other* high-resolution images and applies it to plausibly "fill in" detail in a new low-resolution image.
- **CNN super-resolution** and **PALM/STORM super-resolution microscopy** solve different problems by different mechanisms — a learned prior over many images vs. precise sub-pixel localisation of individually-isolated, sparsely-activated emitters — and should never be conflated just because they share a name.

## ✅ Checkpoint

1. Why does periodic noise show up so clearly in the Fourier domain, and how does that help remove it?
   <details><summary>Answer</summary>A spatial sinusoidal pattern corresponds to a small number of sharp, concentrated peaks in the Fourier spectrum (rather than being spread broadly like Gaussian noise), so those specific frequency peaks can be located and suppressed (e.g. with a notch filter) with far less collateral blurring of the rest of the image than a blanket low-pass filter would cause.</details>
2. What specifically goes wrong if you train a plain (non-denoising) autoencoder to reconstruct its own clean input?
   <details><summary>Answer</summary>It can trivially learn the identity function (just copy input to output) without learning anything about what structure is "real" signal versus noise, so it provides no useful denoising or representation-learning benefit.</details>
3. In what precise sense can single-image super-resolution never "recover lost information"?
   <details><summary>Answer</summary>Any processing applied to a single already-captured low-resolution image can only use the information already present in that image (the data-processing inequality); any fine detail beyond that must come from an external source, such as a prior learned from many other high-resolution images, which is an informed guess rather than a true measurement.</details>
4. Why can PALM/STORM achieve a spatial resolution far finer than the microscope's diffraction limit, while still being governed by the same diffraction-limited PSF as any other fluorescence image?
   <details><summary>Answer</summary>Each individual emitter's image is still diffraction-blurred, but as long as emitters are sparse enough not to overlap, the CENTRE of that blurred spot can be localised to a small fraction of the blur width by fitting its shape (e.g. a 2D Gaussian) — resolution here is limited by localisation precision, not directly by the PSF width.</details>

## 🧠 Questions & discussion

- The denoising CAE in Section 2 was trained and evaluated on the *same* noise type and noise level (Gaussian, σ = 0.12) it was trained on. What would you expect to happen to its performance if you evaluated it on heavy salt-and-pepper noise instead, and why?
- SRCNN's loss function (Section 4) directly minimises pixelwise MSE. Section 3 noted that SSIM sometimes better matches human perceptual judgement. What would likely change about SRCNN's output if it were trained to minimise `1 - SSIM` instead of MSE?
- The PALM/STORM toy demo in Section 5 assumed every emitter's spot was fully isolated. Sketch, in words, what would have to change in the localisation step if two emitters' spots partially overlapped in a given frame.

## 🚀 Bonus — going further

- Add Poisson (shot) noise (`add_poisson_noise` from Module 02's additions to the toolkit, if available in your copy, or implement it directly) to the denoising comparison in Section 2 — does the ranking of the four methods change?
- Make the denoising CAE deeper (add one more conv layer each to encoder and decoder) and see whether that measurably improves held-out PSNR/SSIM, or whether the extra capacity starts to overfit the 12 training images.
- In the PALM/STORM toy demo, generate a "clustered" point pattern (`pattern="clustered"`) instead of `"dispersed"` and see at what cluster tightness the Gaussian-fitting localisation starts to break down as emitters' spots begin to overlap.

## 📝 Lab note

```text
Module 16 — Denoising and super-resolution
Data: SIMULATED nuclei images, course_toolkit.simulate_nuclei_image(seed=16)

Denoising (held-out, Gaussian noise sigma=0.12):
  noisy          PSNR [ ] dB  SSIM [ ]
  mean filter    PSNR [ ] dB  SSIM [ ]
  FFT low-pass   PSNR [ ] dB  SSIM [ ]
  wavelet        PSNR [ ] dB  SSIM [ ]
  denoising CAE  PSNR [ ] dB  SSIM [ ]

Super-resolution (held-out, downscale=2):
  bicubic   PSNR [ ] dB  SSIM [ ]
  SRCNN     PSNR [ ] dB  SSIM [ ]

PALM/STORM toy: PSF sigma [ ] px, mean localisation error [ ] px
Notes:
```

## 📚 References

1. Vincent, P., Larochelle, H., Bengio, Y., & Manzagol, P.-A. (2008). Extracting and composing robust features with denoising autoencoders. *ICML*.
2. Donoho, D. L., & Johnstone, I. M. (1994). Ideal spatial adaptation by wavelet shrinkage. *Biometrika*.
3. Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). Image quality assessment: from error visibility to structural similarity. *IEEE Transactions on Image Processing*.
4. Dong, C., Loy, C. C., He, K., & Tang, X. (2016). Image super-resolution using deep convolutional networks (SRCNN). *IEEE Transactions on Pattern Analysis and Machine Intelligence*.
5. Betzig, E. et al. (2006). Imaging intracellular fluorescent proteins at nanometre resolution (PALM). *Science*.
6. Rust, M. J., Bates, M., & Zhuang, X. (2006). Sub-diffraction-limit imaging by stochastic optical reconstruction microscopy (STORM). *Nature Methods*.
7. Thompson, R. E., Larson, D. R., & Webb, W. W. (2002). Precise nanometer localization analysis for individual fluorescent probes. *Biophysical Journal*.

## ➡️ Next

[**Module 17 — Restoration and inverse problems**](17_restoration_and_inverse_problems.ipynb): we return to the degradation model that opened this notebook — blur plus noise — and treat restoration formally as an *inverse problem*, including the Wiener filter and a glimpse of learned, content-aware restoration for fluorescence microscopy.